# v7 current-session canary correction
Copy these two cells into the existing notebook. No migration rerun or training.
See docs/DRIVE_CANARY_V7_CURRENT_SESSION.md for stop gates and investigation.

In [ ]:
import hashlib
import importlib
import json
import sys
import urllib.parse
from pathlib import Path

from google.colab import files

assert all(
    name in globals()
    for name in ("DRIVE_RUN_ID", "context", "gold", "EXPECTED_SECOND_ACCOUNT_EMAIL", "safe_extract")
), "STOP: use the current kernel where v6 canary upload completed"
DRIVE_RUN_ID = globals()["DRIVE_RUN_ID"]
context = globals()["context"]
gold = globals()["gold"]
EXPECTED_SECOND_ACCOUNT_EMAIL = globals()["EXPECTED_SECOND_ACCOUNT_EMAIL"]
safe_extract = globals()["safe_extract"]
CONFIG = Path("/content") / (DRIVE_RUN_ID + "-drive-config.json")
assert CONFIG.is_file(), "STOP: retain the original v6 config; do not create a new one"
assert not (Path("/content") / (DRIVE_RUN_ID + "-recovery")).exists(), (
    "STOP: already-prepared recovery needs its existing processor-path binding preserved"
)
V7_HELPERS = Path("/content") / (DRIVE_RUN_ID + "-helpers-v7")
PATCH7 = Path("/content") / (DRIVE_RUN_ID + "-patch-v7.zip")
assert not V7_HELPERS.exists() and not PATCH7.exists(), "Preserve existing v7 attempts"
uploaded = files.upload()  # select ONLY recovery-tools-v7.zip
assert set(uploaded) == {"recovery-tools-v7.zip"}
assert (
    hashlib.sha256(uploaded["recovery-tools-v7.zip"]).hexdigest()
    == "d3626c42ceac5b93dc5ebc1b82c144aba7db1430b90cc669c5ed39784e2b1bd2"
)
with PATCH7.open("xb") as stream:
    stream.write(uploaded["recovery-tools-v7.zip"])
safe_extract(PATCH7, V7_HELPERS)
sys.path.insert(0, str(V7_HELPERS))
importlib.invalidate_caches()
for module_name in ("drive_handoff", "drive_handoff_setup"):
    if module_name in sys.modules:
        module = importlib.reload(sys.modules[module_name])
    else:
        module = importlib.import_module(module_name)
    assert Path(module.__file__).resolve().parent == V7_HELPERS.resolve()
DRIVE_HELPERS = V7_HELPERS  # subsequent existing Cell6 uses the versioned helpers/assets
print("V7_PATCH_LOADED: original v6 files and completed Cells1-4 preserved")

In [ ]:
drive_handoff = sys.modules["drive_handoff"]
drive_handoff_setup = sys.modules["drive_handoff_setup"]
config = json.loads(CONFIG.read_text(encoding="utf-8"))
receipt_path = Path(config["canary_receipt"])
original_config_bytes = CONFIG.read_bytes()
original_receipt_bytes = receipt_path.read_bytes()
receipt = json.loads(original_receipt_bytes)
assert config["run_id"] == DRIVE_RUN_ID
assert receipt["bytes"] == 1_048_694
assert (
    receipt["sha256"]
    == config["canary_sha256"]
    == ("aa6b8846a8fdd5d5ac12d5b8950914f2cd4dddec04146d7cf5b3f1bf0d96127b")
), "STOP: this patch procedure is pinned to the reported existing canary"
if "BROKER_STOP" in globals():
    globals()["BROKER_STOP"].set()
# Refresh current second-account credentials; no folder/file creation or upload.
CONFIG, folder, BROKER_STOP = drive_handoff_setup.resume_existing(
    CONFIG, context, EXPECTED_SECOND_ACCOUNT_EMAIL
)
client = drive_handoff.DriveAPI(config)
assert CONFIG.read_bytes() == original_config_bytes
assert receipt_path.read_bytes() == original_receipt_bytes

# Read-only diagnostics; Drive version is not a content-revision identifier.
# Historical metadata at version1 is unavailable, so this cannot prove its cause.
file_id = urllib.parse.quote(receipt["remote_file_id"], safe="")
diagnostics = {
    "receipt": receipt,
    "config_sha256": hashlib.sha256(original_config_bytes).hexdigest(),
    "exact_server_side_version_trigger": "not established",
}
for label, suffix in (
    (
        "current_metadata",
        "files/"
        + file_id
        + "?"
        + urllib.parse.urlencode(
            {
                "fields": "id,version,headRevisionId,modifiedTime,viewedByMeTime,"
                "lastModifyingUser(permissionId),size,sha256Checksum,md5Checksum,parents,"
                "ownedByMe,owners(permissionId)"
            }
        ),
    ),
    (
        "revision_page",
        "files/"
        + file_id
        + "/revisions?"
        + urllib.parse.urlencode(
            {
                "pageSize": 1000,
                "fields": "nextPageToken,revisions(id,modifiedTime,size,md5Checksum,"
                "lastModifyingUser(permissionId))",
            }
        ),
    ),
):
    try:
        diagnostics[label] = client.json_request("GET", suffix)
    except drive_handoff.PersistenceError as error:
        diagnostics[label] = {"unavailable": str(error)}  # diagnostics only; gate already passed
inspection = CONFIG.with_name(CONFIG.stem + "-v7-inspection.json")
with inspection.open("x", encoding="utf-8") as stream:
    json.dump(diagnostics, stream, indent=2)
print("V7_EXISTING_CANARY_PASS: ID/owner/parent/size/full-SHA/stable-read verified")
print("Preserved receipt:", receipt_path)
print("Revalidation log:", receipt_path.with_suffix(".revalidations.jsonl"))
print("Read-only inspection:", inspection)
print("STOP: no training/evaluation started. Readiness Cell6/7 may follow; do not run old5A/5B.")